# 3. Message Handling in Agentic Models

A detailed exploration of structured message architectures in LangChain:
- **`SystemMessage`**: Setting personas, role definitions, and system-level guidelines
- **`HumanMessage`**: Representing user queries with contextual metadata
- **`AIMessage`**: Capturing model completions and tool call requests
- **`ToolMessage`**: Feeding tool execution results back to the model
- **Multi-Turn Streaming**: Preserving message history during real-time streaming
- **Response Metadata**: Inspecting token usage, finish reasons, and performance metrics

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # reads OLLAMA_MODEL from .env (set per machine)
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_ollama import ChatOllama

model = ChatOllama(model=os.getenv("OLLAMA_MODEL", "qwen3:8b"), temperature=0, reasoning=False)

messages = [
    SystemMessage(
        """You are an AI expert with experience in UI/UX development and REST APIs with Java and Python.
You are also an expert in Linux systems.
Provide structured solutions with clear architectural explanations."""
    ),
    AIMessage("Hello! I am your AI assistant. How may I help you with your architecture today?"),
    HumanMessage(
        content="What is the best way to use a custom UI for local LLMs like Ollama?",
        name="Mandar",
        id="1"
    )
]

resp = model.invoke(messages)
print(resp.text)

### 1. Multi-Turn Conversation History with Structured Messages

- **`SystemMessage`**: Injects persistent persona and domain context into the prompt template.
- **`AIMessage`**: Feeds prior assistant utterances to preserve conversational continuity.
- **`HumanMessage`**: Represents the user's latest query along with user metadata (`name`, `id`).
- **Context-Aware Completion**: Generates responses conditioned on the full dialogue history.

In [ ]:
for chunk in model.stream(messages):
    print(chunk.text, end="|", flush=True)

### 2. Streaming Multi-Turn Conversations

- **Streaming Multi-Turn Context**: Passes the full `messages` list to `model.stream()`.
- **Interactive Feedback**: Streams token responses in real time without sacrificing multi-turn conversational context.

In [ ]:
ai_message = AIMessage(
    content="",
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "Pune"},
        "id": "call_abc123"
    }]
)

tool_message = ToolMessage(
    content="Sunny, 26°C with light breeze",
    tool_call_id="call_abc123"
)

agentic_messages = [
    HumanMessage("What is the weather in Pune?"),
    ai_message,
    tool_message
]

tool_resp = model.invoke(agentic_messages)
print(tool_resp.text)

### 3. Synthetic Tool Message Sequences

- **Agentic State Representation**: Demonstrates how agent state graphs represent tool execution in memory.
- **Call-Response Pairing**: Connects `AIMessage(tool_calls=[...])` with `ToolMessage(tool_call_id=...)` via matching IDs.
- **Grounding**: The LLM reads the tool output from `ToolMessage` and generates a verified response.

In [ ]:
import pprint

print("Response Metadata:")
pprint.pprint(tool_resp.response_metadata)

### 4. Inspecting LLM Response Metadata

- **Metadata Inspection**: Accesses `resp.response_metadata` to retrieve diagnostic information.
- **Execution Diagnostics**: Includes token counts, model identifiers, completion reasons, and latency metrics.